In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
import numpy as np
import os
import cv2
from glob import glob
from tensorflow.keras import layers # to use custom layers , we used custom activation function, also skip connections to preseerve details
import random # to shuffle dataset as images are taken from an online  dataset there may be patterns so we shuffle
#dataset so that our model can be trained better



In [ ]:
# Parameters
IMAGE_SIZE = 128
BATCH_SIZE = 4
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 25

imgpth = "/content/drive/MyDrive/Humans"

import os

print("Checking dataset path...")
print(os.listdir(imgpth)[:5])

image_paths = glob(os.path.join(imgpth, "*.jpg")) + glob(os.path.join(imgpth, "*.png")) + glob(os.path.join(imgpth, "*.jpeg"))
print("Total images found:", len(image_paths))


def preprocess(path):
    path = path.numpy().decode("utf-8")  # convert tensor → string

    image = cv2.imread(path)
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    sketch = cv2.Canny(gray, 100, 200)
    sketch = np.expand_dims(sketch, axis=-1)

    image = image / 255.0
    sketch = sketch / 255.0

    return sketch.astype(np.float32), image.astype(np.float32)

def tf_preprocess(path):
    sketch, image = tf.py_function(func=preprocess, inp=[path], Tout=[tf.float32, tf.float32])
    sketch.set_shape([IMAGE_SIZE, IMAGE_SIZE, 1])
    image.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3])
    return sketch, image

random.shuffle(image_paths)

dataset = tf.data.Dataset.from_tensor_slices(image_paths)
dataset = dataset.map(tf_preprocess, num_parallel_calls=AUTOTUNE)
dataset = dataset.shuffle(1000)
dataset = dataset.batch(BATCH_SIZE)
dataset = dataset.prefetch(AUTOTUNE)






Checking dataset path...
['1 (6481).jpg', '1 (6431).jpg', '1 (6473).jpg', '1 (6452).jpg', '1 (6415).jpg']
Total images found: 7202


In [ ]:
# Sketch generation (using Canny edge)
def create_sketch(image_np):
    if image_np.shape[-1] == 3:
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    else:
        gray = image_np[..., 0]

    if gray.dtype != np.uint8:
        gray = (gray * 255).astype(np.uint8) if gray.max() <= 1.0 else gray.astype(np.uint8)

    edges = cv2.Canny(gray, threshold1=100, threshold2=200)
    sketch = edges.astype(np.float32) / 255.0
    sketch = np.expand_dims(sketch, axis=-1)
    return sketch
# in input also we have to use canny sketches even if user givves hand drawn sketches aswe are training over
#edge detected sketches




In [ ]:
# Preprocess function
def preprocess(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image_np = image.numpy()
    sketch = create_sketch(image_np)

    sketch = (sketch * 2.0) - 1.0 # normalise pixel values to use activation function, we use tanh activation for GAN
    image = (image_np / 127.5) - 1.0
    return sketch.astype(np.float32), image.astype(np.float32)

def tf_preprocess(path):
    sketch, img = tf.py_function(preprocess, [path], [tf.float32, tf.float32])
    sketch.set_shape([IMAGE_SIZE, IMAGE_SIZE, 1])
    img.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3])
    return sketch, img


dataset = tf.data.Dataset.from_tensor_slices(image_paths)
dataset = dataset.map(tf_preprocess, num_parallel_calls=AUTOTUNE)
dataset = dataset.shuffle(100).batch(BATCH_SIZE).prefetch(AUTOTUNE)



In [ ]:
# Generator & Discriminator

#generator
def downsample(filters, size, apply_batchnorm=True):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2D(filters, size, strides=2, padding='same',
                      kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

def upsample(filters, size, apply_dropout=False):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2DTranspose(filters, size, strides=2,
                               padding='same',
                               kernel_initializer=initializer,
                               use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def Generator():
    inputs = layers.Input(shape=[IMAGE_SIZE, IMAGE_SIZE, 1])
    down_stack = [
        downsample(64, 4, apply_batchnorm=False),
        downsample(128, 4),
        downsample(256, 4),
        downsample(512, 4),
    ]
    up_stack = [
        upsample(512, 4, apply_dropout=True),
        upsample(256, 4),
        upsample(128, 4),
        upsample(64, 4),
    ]
    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4,
                                  strides=2,
                                  padding='same',
                                  kernel_initializer=initializer,
                                  activation='tanh')  # [-1, 1] normalised pixel values already in sketches

    x = inputs
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)
    skips = reversed(skips[:-1])

    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = last(x)
    return tf.keras.Model(inputs=inputs, outputs=x)

#discriminator

def Discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)
    inp = layers.Input(shape=[IMAGE_SIZE, IMAGE_SIZE, 1], name='input_image')
    tar = layers.Input(shape=[IMAGE_SIZE, IMAGE_SIZE, 3], name='target_image')
    x = layers.Concatenate()([inp, tar])
    down1 = downsample(64, 4, False)(x)
    down2 = downsample(128, 4)(down1)
    down3 = downsample(256, 4)(down2)
    zero_pad1 = layers.ZeroPadding2D()(down3)
    conv = layers.Conv2D(512, 4, strides=1, kernel_initializer=initializer, use_bias=False)(zero_pad1)
    batchnorm1 = layers.BatchNormalization()(conv)
    leaky_relu = layers.LeakyReLU()(batchnorm1)
    zero_pad2 = layers.ZeroPadding2D()(leaky_relu)
    last = layers.Conv2D(1, 4, strides=1, kernel_initializer=initializer)(zero_pad2)
    return tf.keras.Model(inputs=[inp, tar], outputs=last)




In [ ]:
# Losses computation
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_obj(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_obj(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss

def generator_loss(disc_generated_output, gen_output, target):
    gan_loss = loss_obj(tf.ones_like(disc_generated_output), disc_generated_output)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    total_gen_loss = gan_loss + (100 * l1_loss)
    return total_gen_loss



In [ ]:

# Training  function
generator = Generator()
discriminator = Discriminator()

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))
    return gen_loss, disc_loss



In [ ]:
import os
import tensorflow as tf

checkpoint_dir = './checkpoints' # saving checkpoints we will finally load the lasst model although
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

checkpoint = tf.train.Checkpoint(generator=generator,
                                 discriminator=discriminator,
                                 generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer)

manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir, max_to_keep=5)


In [ ]:
SAVE_EVERY_N_STEPS = 200

for epoch in range(EPOCHS): # training loop
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    for step, (sketch, real_image) in enumerate(dataset):
        gen_loss, disc_loss = train_step(sketch, real_image)

        if step % 100 == 0:
            print(f"Step {step}: Gen Loss = {gen_loss:.4f}, Disc Loss = {disc_loss:.4f}")

        if step % SAVE_EVERY_N_STEPS == 0 and step > 0:
            save_path = manager.save()
            print(f"✅ Checkpoint saved at step {step} -> {save_path}")



Epoch 1/25
Step 0: Gen Loss = 54.1503, Disc Loss = 1.8389
Step 100: Gen Loss = 51.4269, Disc Loss = 0.1545
Step 200: Gen Loss = 59.8391, Disc Loss = 0.0900
✅ Checkpoint saved at step 200 -> ./checkpoints/ckpt-1
Step 300: Gen Loss = 52.8836, Disc Loss = 0.1703
Step 400: Gen Loss = 54.1828, Disc Loss = 1.3747
✅ Checkpoint saved at step 400 -> ./checkpoints/ckpt-2
Step 500: Gen Loss = 48.4637, Disc Loss = 0.6507
Step 600: Gen Loss = 53.6181, Disc Loss = 1.1295
✅ Checkpoint saved at step 600 -> ./checkpoints/ckpt-3
Step 700: Gen Loss = 52.8564, Disc Loss = 1.1172
Step 800: Gen Loss = 49.5543, Disc Loss = 0.8289
✅ Checkpoint saved at step 800 -> ./checkpoints/ckpt-4
Step 900: Gen Loss = 45.5151, Disc Loss = 0.7878
Step 1000: Gen Loss = 51.5844, Disc Loss = 0.4161
✅ Checkpoint saved at step 1000 -> ./checkpoints/ckpt-5
Step 1100: Gen Loss = 43.0534, Disc Loss = 2.5983
Step 1200: Gen Loss = 48.0829, Disc Loss = 0.5505
✅ Checkpoint saved at step 1200 -> ./checkpoints/ckpt-6
Step 1300: Gen Los

In [ ]:
#final saving of model
generator.save("generator_sketch2image.h5")
